# EXACT Phase‑1 v40.4 Replay / Hardening Notebook

Purpose: replay the **real BTC Phase‑1 Type1 logs** with the conservative `v40.4` entity-grounded conjunctive Horn engine.

## Inputs

Required, auto-detected:
- `exact_eval_round1_Astatine.json`

Search locations:
- `/kaggle/working/**`
- `/kaggle/input/**`
- `/mnt/data/**`
- current directory

## Outputs

- `/kaggle/working/v40_4_entity_conjunctive_engine.py`
- `/kaggle/working/v40_4_phase1_replay_report.json`
- `/kaggle/working/v40_4_phase1_replay_cases.json`

Gate: accept only when `wrong == []`, `answer_correct == fired`, and `premises_used_correct` does not regress.

In [ ]:
# CELL 0 — write v40.4 engine to /kaggle/working
from pathlib import Path
import json, os, glob, sys, textwrap, subprocess, re
WORK = Path('/kaggle/working') if Path('/kaggle').exists() else Path('/mnt/data')
WORK.mkdir(parents=True, exist_ok=True)
ENGINE_PATH = WORK / 'v40_4_entity_conjunctive_engine.py'
ENGINE_PATH.write_text('# -*- coding: utf-8 -*-\n"""v40.3 entity-grounded conjunctive Horn engine for the REAL Phase-1 distribution.\nPropositional over a single entity: facts = literals, rules = (conj of literals)->literal.\nForward-chain; answer options by derivability. Certificate = premise indices. Abstain-safe."""\nimport re\nSTOP={\'a\',\'an\',\'the\',\'of\',\'to\',\'in\',\'on\',\'at\',\'for\',\'and\',\'or\',\'that\',\'this\',\'their\',\'its\',\'it\',\'they\',\'them\',\n      \'is\',\'are\',\'was\',\'were\',\'be\',\'been\',\'has\',\'have\',\'had\',\'then\',\'if\',\'no\',\'not\',\'with\',\'as\',\'by\',\'from\',\n      \'artifact\',\'package\',\'manuscript\',\'sample\',\'batch\',\'item\',\'device\',\'record\',\'file\',\'student\'}\ndef _stem(t):\n    if re.search(r\'(ss|us|is)$\',t): return t\n    if re.search(r\'(ches|shes|xes|zes|ses)$\',t): return t[:-2]\n    if re.search(r\'ies$\',t): return t[:-3]+\'y\'\n    if t.endswith(\'s\'): t=t[:-1]\n    return re.sub(r\'(ing|ed)$\',\'\',t)\ndef atom_key(phrase):\n    s=re.sub(r\'(?<!^)(?=[A-Z])\',\' \',str(phrase)).lower()\n    nums=re.findall(r\'\\d+\', s)\n    toks=[ _stem(w) for w in re.findall(r\'[a-zA-Z]+\', s) ]\n    toks=[t for t in toks if t and t not in STOP and len(t)>2]\n    keys=sorted(set(toks))+["N"+n for n in sorted(set(nums))]   # keep numeric literals\n    return "".join(w.capitalize() for w in keys) if keys else None\n\n# split a clause into (atom, neg). Handles "X has Y", "X is Y", "X has no Y", "X lacks Y", "cannot ...", "is not ..."\n_LEAD=re.compile(r"^\\s*(if|then|that|who|which|it|its|their|this)\\b",re.I)\n_VERB=re.compile(r"\\b(cannot|can not|can|could|may|might|must|should|shall|will|would|is not|are not|was not|were not|isn\'t|aren\'t|is|are|was|were|has no|have no|had no|has|have|had|lacks?|without|requires?|needs?|contains?|completed?|enters?|gains?|receives?|provides?|shows?|states?|holds?|carries|monitors?|captures?|eligible|allowed|approved|assigned|be|been|being)\\b",re.I)\nACTION_VERBS={\'receives\',\'receive\',\'provides\',\'provide\',\'shows\',\'show\',\'states\',\'state\',\'monitors\',\'monitor\',\'captures\',\'capture\',\'enters\',\'enter\',\'requires\',\'require\',\'needs\',\'need\',\'gains\',\'gain\',\'completed\',\'complete\',\'contains\',\'contain\',\'reports\',\'report\',\'releases\',\'release\',\'passes\',\'pass\',\'improves\',\'improve\',\'supports\',\'support\',\'recommends\',\'recommend\',\'administered\',\'administer\',\'approved\',\'approve\'}\n_NEGWORD=re.compile(r\'\\b(no|not|cannot|can not|lacks?|without|isn\\\'t|aren\\\'t|never|nor|incomplete|missing|lacking)\\b\',re.I)\ndef to_literal(clause):\n    c=clause.strip().rstrip(\'.\').strip()\n    c=_LEAD.sub(\'\',c).strip()\n    neg=bool(re.search(r"\\b(no|not|cannot|can not|never|lacks?|without|isn\'t|aren\'t|incomplete|missing|lacking|nor|un(?:able|verified|established))\\b",c,re.I))\n    m=_VERB.search(c)\n    pred=c[m.end():].strip() if m else c\n    verb=(m.group(1).lower() if m else \'\')\n    if m and verb in ACTION_VERBS:\n        pred = verb + \' \' + pred\n    # peel any leftover leading modal/aux/passive markers and articles\n    for _ in range(4):\n        pred=re.sub(r"^\\s*(be|been|being|to|a|an|the|no|not|its|their)\\b","",pred,flags=re.I).strip()\n    a=atom_key(pred)\n    return (a,neg) if a else None\n\ndef parse_premise(p):\n    s=str(p).strip()\n    m=re.search(r\'^\\s*if\\b(.+?),?\\s*\\bthen\\b(.+)$\',s,re.I)\n    if m:\n        ante=re.split(r\'\\band\\b\',m.group(1),flags=re.I)\n        lits=[to_literal(x) for x in ante]; lits=[l for l in lits if l]\n        con=to_literal(m.group(2))\n        if con and lits: return (\'rule\',lits,con)\n        return None\n    # Universal relative rule: Every/All <role> <condition> <verb> <consequent>\n    # Examples: Every volunteer assigned to shift receives badge; All satellites with cameras can capture images.\n    m2=re.search(r\'^\\s*(every|all)\\s+[a-zA-Z]+s?\\s+(.+?)\\s+\\b(can|may|must|should|will|would|receives?|gets?|gains?|provides?|captures?|monitors?|requires?|needs?|is|are)\\b\\s+(.+)$\',s,re.I)\n    if m2:\n        cond=m2.group(2).strip()\n        cons=(m2.group(3)+" "+m2.group(4)).strip()\n        litc=to_literal(cond); litd=to_literal(cons)\n        if litc and litd: return (\'rule\',[litc],litd)\n    if re.search(r\'^\\s*(no premise|it (is|cannot)|unknown|there is no information)\',s,re.I): return None\n    lit=to_literal(s)\n    return (\'fact\',lit) if lit else None\n\ndef solve_entity(premises):\n    facts={}  # atom -> (bool_value, premise_idx)\n    rules=[]\n    for i,p in enumerate(premises):\n        pp=parse_premise(p)\n        if not pp: continue\n        if pp[0]==\'fact\':\n            a,neg=pp[1]; facts.setdefault(a,(not neg,[i]))\n        else:\n            rules.append((i,pp[1],pp[2]))\n    changed=True\n    while changed:\n        changed=False\n        for i,lits,con in rules:\n            ca,cneg=con\n            ok=True; path=[i]\n            for a,neg in lits:\n                if a in facts and facts[a][0]==(not neg): path+=facts[a][1]\n                else: ok=False; break\n            if ok and ca not in facts:\n                facts[ca]=((not cneg),sorted(set(path))); changed=True\n    return facts\n\n_META_RE=__import__("re").compile(r"\\b(not (?:yet )?(?:established|confirmed|verified|approved|cleared|determined)|cannot be (?:established|confirmed)|unsupported|is not established|no premise|undetermined|not (?:available|present))\\b",__import__("re").I)\ndef decompose_option(opt):\n    import re\n    t=re.sub(r\'^\\s*[A-Da-d][.):]\\s*\',\'\',str(opt)).strip()\n    t=re.split(r\'\\bbecause\\b\', t, maxsplit=1, flags=re.I)[0].strip()  # drop causal justification\n    parts=re.split(r\',\\s*but\\s+|\\s+but\\s+|;\\s+|\\s+while\\s+|\\s+whereas\\s+|\\s+and\\s+\',t,flags=re.I)\n    claims=[]\n    for p in parts:\n        p=p.strip()\n        if not p: continue\n        is_meta=bool(_META_RE.search(p))\n        lit=to_literal(p)\n        claims.append((lit,is_meta,p))\n    return claims\n\ndef answer_mc(premises, options):\n    facts=solve_entity(premises)\n    res={}\n    for lab,opt in zip("ABCD",options):\n        claims=decompose_option(opt)\n        if not claims: res[lab]=(\'UNSUP\',[]); continue\n        status=\'PROVEN\'; path=[]\n        for lit,is_meta,txt in claims:\n            if lit is None: status=\'UNSUP\'; break\n            a,neg=lit; have = a in facts\n            val = facts[a][0] if have else None\n            if is_meta:\n                # meta "not established": correct only if NOT positively proven\n                if have and val==True: status=\'DISPROVEN\'; break\n            else:\n                if have and val==(not neg): path+=facts[a][1]\n                elif have and val==(neg): status=\'DISPROVEN\'; break\n                else: status=\'UNSUP\'; break\n        res[lab]=(status, sorted(set(path)))\n    proven=[l for l in res if res[l][0]==\'PROVEN\']\n    if len(proven)==1: return proven[0],res[proven[0]][1],\'entity_unique_proof\',res\n    return None,[],(\'multiple\' if proven else \'none\'),res\n\n\n# ================= Phase-1 REPLAY HARNESS =================\ndef _opt_texts(rp):\n    import re\n    f=[o[1].strip().replace("\\n"," ") for o in re.findall(r"(?:^|\\n)\\s*([A-D])[.)]\\s*(.+?)(?=\\n\\s*[A-D][.)]|\\Z)",rp.get("query",""),flags=re.S)]\n    return f if len(f)>=2 else (rp.get("options") or [])\ndef replay_phase1(path):\n    import json,re\n    d=json.load(open(path)); t1=[l for l in d["logs"] if l.get("type")=="type1"]\n    fired=aok=pok=0; fixed=[]; wrong=[]; abst=0\n    for l in t1:\n        rp=l["request_payload"]; exp=l.get("expected") or {}; ea=str(exp.get("answer","")).strip().upper()\n        opts=_opt_texts(rp)\n        if not opts: continue\n        a,pu,why,res=answer_mc(rp.get("premises",[]) or [], opts)\n        if a is None: abst+=1; continue\n        fired+=1; ok=(a==ea); aok+=ok; pok+=(sorted(pu)==sorted(exp.get("premises_used") or []))\n        if ok and l.get("status")!="correct": fixed.append(l["query_id"])\n        if not ok: wrong.append({"query_id":l["query_id"],"exp":ea,"got":a})\n    rep={"n":len(t1),"fired":fired,"answer_correct":aok,"premises_used_correct":pok,\n         "wrong":wrong,"fixed_old_wrong":fixed,"abstained":abst,\n         "precision_when_fired":round(aok/max(fired,1),3),"coverage":round(fired/max(len(t1),1),3),\n         "gate":"ABSTAIN_SAFE" if not wrong else "HAS_WRONG_FIX_BEFORE_APPLY"}\n    return rep\ndef _autofind():\n    import glob,os,sys\n    if len(sys.argv)>1 and os.path.exists(sys.argv[1]): return sys.argv[1]\n    for c in ["exact_eval_round1_Astatine.json","/kaggle/input/**/exact_eval_round1_Astatine.json",\n              "/kaggle/working/exact_eval_round1_Astatine.json","./exact_eval_round1_Astatine.json"]:\n        h=sorted(glob.glob(c,recursive=True)) if any(x in c for x in "*?[") else ([c] if os.path.exists(c) else [])\n        if h: return h[0]\n    return None\nif __name__=="__main__":\n    import json\n    p=_autofind()\n    if not p: print("exact_eval_round1_Astatine.json not found; pass path as arg."); raise SystemExit(1)\n    rep=replay_phase1(p); json.dump(rep,open("v40_phase1_replay_report.json","w"),indent=1)\n    print(json.dumps(rep,indent=1))\n', encoding='utf-8')
print('Wrote:', ENGINE_PATH)
print('Size:', ENGINE_PATH.stat().st_size)

In [ ]:
# CELL 1 — auto-find exact_eval_round1_Astatine.json
from pathlib import Path
import glob, os, json
SEARCH_PATTERNS = [
    '/kaggle/working/**/exact_eval_round1_Astatine.json',
    '/kaggle/input/**/exact_eval_round1_Astatine.json',
    '/mnt/data/**/exact_eval_round1_Astatine.json',
    './exact_eval_round1_Astatine.json',
]
CANDIDATES=[]
for pat in SEARCH_PATTERNS:
    CANDIDATES += glob.glob(pat, recursive=True)
CANDIDATES = sorted(set([p for p in CANDIDATES if os.path.exists(p)]))
print('Candidates:')
for p in CANDIDATES:
    print(' -', p)
if not CANDIDATES:
    raise FileNotFoundError('exact_eval_round1_Astatine.json not found. Attach/upload it or add it as a Kaggle dataset.')
PHASE1_PATH = CANDIDATES[0]
print('Using PHASE1_PATH =', PHASE1_PATH)

In [ ]:
# CELL 2 — run v40.4 replay and save report
import importlib.util, json, os
spec = importlib.util.spec_from_file_location('v40_4', str(ENGINE_PATH))
v40 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(v40)
rep = v40.replay_phase1(PHASE1_PATH)
print(json.dumps(rep, indent=2, ensure_ascii=False))
OUT = WORK / 'v40_4_phase1_replay_report.json'
OUT.write_text(json.dumps(rep, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved:', OUT)
assert rep['wrong'] == [], 'Gate FAIL: v40.4 produced wrong answers'
assert rep['answer_correct'] == rep['fired'], 'Gate FAIL: not all fired answers are correct'
print('GATE PASS:', rep.get('gate'))

In [ ]:
# CELL 3 — detailed fired/abstained cases for review
import json, re
from pathlib import Path

def opt_texts(rp):
    f=[o[1].strip().replace('
',' ') for o in re.findall(r"(?:^|
)\s*([A-D])[.)]\s*(.+?)(?=
\s*[A-D][.)]|\Z)",rp.get('query',''),flags=re.S)]
    return f if len(f)>=2 else (rp.get('options') or [])

d=json.load(open(PHASE1_PATH, encoding='utf-8'))
rows=[]
for l in [x for x in d['logs'] if x.get('type')=='type1']:
    rp=l['request_payload']; exp=l.get('expected') or {}
    opts=opt_texts(rp)
    a,pu,why,res=v40.answer_mc(rp.get('premises',[]) or [], opts)
    rows.append({
        'query_id': l.get('query_id'),
        'old_status': l.get('status'),
        'expected_answer': exp.get('answer'),
        'expected_premises_used': exp.get('premises_used'),
        'v40_answer': a,
        'v40_premises_used': pu,
        'rule': why,
        'answer_ok': (a is not None and str(a).upper()==str(exp.get('answer','')).upper()),
        'premises_ok': (a is not None and sorted(pu)==sorted(exp.get('premises_used') or [])),
        'query': rp.get('query','')[:300],
        'option_status': res,
    })
OUT2 = WORK / 'v40_4_phase1_replay_cases.json'
OUT2.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved:', OUT2)
print('Fired cases:')
for r in rows:
    if r['v40_answer'] is not None:
        print(r['query_id'], 'old=', r['old_status'], 'exp=', r['expected_answer'], 'got=', r['v40_answer'], 'prem_ok=', r['premises_ok'])
print('
First abstains:')
for r in rows:
    if r['v40_answer'] is None:
        print(r['query_id'], 'old=', r['old_status'], 'rule=', r['rule'], 'q=', r['query'][:100])
        if sum(1 for x in rows[:rows.index(r)+1] if x['v40_answer'] is None) >= 8:
            break

## Interpretation

- If `wrong=[]`, v40.4 is safe as an abstain-first Type1 MC pre-handler.
- If `fired` increases in future patches but `wrong` becomes non-empty, rollback the patch.
- This notebook intentionally evaluates on **real Phase‑1 BTC logs**, not generated synthetic.